# Uji CLI end-to-end dan pytest

Clone repo, unduh semua yang tidak dilacak Git (checkpoint NER, checkpoint classifier demo, CSV Komdigi), bangun basis klaim terbantah, lalu jalankan CLI dan `pytest`. Setelah sel 3 dan 4 selesai, tidak ada tahap yang berjalan sebagai `[STUB]`: NER, klasifikasi, retrieval (E5), dan stance (mDeBERTa) memakai model sungguhan.

Runtime GPU tidak wajib; di CPU tiap pemanggilan CLI butuh sekitar satu menit karena model dimuat ulang.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "main"
# Diekstrak berurutan; zip kedua menimpa ner_indobert_claim_seed42 (CLAIM + augmentasi informal RQ1)
NER_ZIP_FILE_IDS = [
    "1c_ouOXwCIdMgqD2_rJzF_oXL-5qS0oEN",  # hasil_training/ner_20261112-1801: entitas + CLAIM lama
    "1s5mZu7px3fsNGz1_BtCq66cl7nl6k0Uv",  # hasil_training/ner_20261114-1153: hanya CLAIM baru
]
# hasil_training/indobert_classifier_seed13.zip: latih ulang seed 13, khusus demo CLI (bukan sumber angka laporan)
CLASSIFIER_ZIP_FILE_ID = "1J2rY1TzUavklp4V0X7zoIHAnNes3f5NF"
# data/archive/komdigi_hoaks.csv di Drive tim (Kaggle ireddragonicy/indonesian-hoax-news-dataset v396)
KOMDIGI_FILE_ID = "1nsRT_9zvGbizx8H-sBSbDG1-vtyfPsp_"

from google.colab import auth
auth.authenticate_user()

## 2. Clone dan install dependensi

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -3
!pip install -q -e ".[training]" pytest

## 3. Download dan ekstrak checkpoint (NER dan classifier)

In [ ]:
import io, json, os, zipfile
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive = build("drive", "v3")

def drive_download(file_id, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with io.FileIO(path, "wb") as f:
        dl = MediaIoBaseDownload(f, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        done = False
        while not done:
            _, done = dl.next_chunk()
    return path

for i, file_id in enumerate(NER_ZIP_FILE_IDS):
    z = zipfile.ZipFile(drive_download(file_id, f"/content/ner_checkpoints_{i}.zip"))
    names = z.namelist()
    print(file_id, len(names), "berkas; contoh:", names[:4])
    z.extractall("." if any(n.startswith("artifacts/") for n in names) else "artifacts/checkpoints/ner")
!ls artifacts/checkpoints/ner
# Model CLAIM baru dilatih dengan augmentasi informal (augment.copies = 3)
cfg = json.load(open("artifacts/checkpoints/ner/ner_indobert_claim_seed42/train_config.json"))
assert cfg.get("augment", {}).get("copies") == 3, "model CLAIM masih versi lama; cek zip kedua"

# Classifier demo: isi zip sudah berjalur artifacts/checkpoints/classifier/indobert_seed13/
zipfile.ZipFile(drive_download(CLASSIFIER_ZIP_FILE_ID, "/content/indobert_classifier_seed13.zip")).extractall(".")
!ls artifacts/checkpoints/classifier/indobert_seed13

## 4. Basis klaim terbantah (wajib untuk retrieval)
`data/processed/` tidak dilacak Git, jadi `claims.jsonl` dibangun ulang dari CSV Komdigi. Tanpa sel ini retrieval dan stance berjalan sebagai stub. Hash hasilnya dicocokkan dengan `data/splits/manifest.json` (dibekukan 12 Nov 2026); keluaran split diarahkan ke folder sementara supaya manifest di repo tidak berubah.

In [ ]:
import hashlib
drive_download(KOMDIGI_FILE_ID, "data/raw/refuted_claims/komdigi_hoaks.csv")
!python -m indonesian_hoax_checker.retrieval.corpus data/raw/refuted_claims/komdigi_hoaks.csv data/interim/refuted_claims/claims_raw.jsonl | tail -3
!python -m indonesian_hoax_checker.evaluation.splits data/interim/refuted_claims/claims_raw.jsonl --kaggle-version 396 --splits-dir /content/splits_tmp | tail -3
CLAIMS = "data/processed/refuted_claims/claims.jsonl"
frozen = {f["file"]: f["sha256"] for f in json.load(open("data/splits/manifest.json"))["files"]}
got = hashlib.sha256(open(CLAIMS, "rb").read()).hexdigest()
assert got == frozen[CLAIMS], f"basis klaim berbeda dari manifest beku: {got}"
print("basis klaim cocok dengan manifest beku:", got[:16])
!git status --short data/splits

## 5. CLI: seluruh pipeline
Pemanggilan pertama mengunduh dua model HF (E5 dan mDeBERTa, masing-masing sekitar 1,1 GB) dan menghitung embedding 3.048 klaim. Tidak boleh ada tanda `[STUB]` di keluaran. Teks di bawah 200 kata sengaja tidak dinilai classifier (`classification.min_words`), jadi barisnya tertulis "dilewati".

In [ ]:
!python -m indonesian_hoax_checker "Aturan baru Pertamina batasi pembelian BBM di semua SPBU" -v
!python -m indonesian_hoax_checker "Tidak benar bahwa air nanas panas dapat menyembuhkan kanker; belum ada bukti ilmiahnya." -v
!python -m indonesian_hoax_checker "Pemerintah Kota Bandung akan membangun taman baca di setiap kelurahan pada tahun 2027." -v
!python -m indonesian_hoax_checker "Pemerintah menghentikan bantuan" --task ner
# Semua kasus demo lewat mode interaktif (model dimuat sekali)
!grep -vE '^(#|$)' data/demo_cases.txt | grep -v '<kosong>' | sed 's/\\n/ /g' | python -m indonesian_hoax_checker | grep -c STUB || true

## 6. Kueri retrieval dan klaim stance (keduanya teks input penuh; core_claim hanya ditampilkan)

In [ ]:
import sys
sys.path.insert(0, "/content/repo/src")   
from indonesian_hoax_checker.pipeline.orchestrator import run_pipeline, retrieval_query, input_claim
text = "Beredar pesan berantai yang mengeklaim Menteri Keuangan Purbaya akan menaikkan harga BBM subsidi pada 1 Desember 2026."
res = run_pipeline(text, "all")
print("stub stages:", res.stubs)
print("entitas   :", [(e["text"], e["label"]) for e in res.ner["entities"]])
print("core_claim:", repr(res.ner["core_claim"]))
print("kueri     :", repr(retrieval_query(text, res.ner)))
print("klaim stance:", repr(input_claim(text, res.ner)))

## 7. pytest

In [ ]:
!python -m pytest -q --no-header -p no:cacheprovider 2>&1 | tail -25
# Uji end-to-end dengan model sungguhan (74 kasus tricky: negasi, entitas ditukar, teks panjang, input aneh, CLI dari folder lain)
!HOAX_CHECKER_E2E=1 python -m pytest tests/integration/test_cli_end_to_end.py -q --no-header -p no:cacheprovider 2>&1 | tail -15

## 8. Classifier IndoBERT di CLI
Checkpoint sudah diekstrak di sel 3. Classifier hanya menilai teks 200 kata atau lebih.

In [ ]:
!ls artifacts/checkpoints/classifier/indobert_seed13
!git status --short configs/pipeline.yaml
!grep -n -A4 "^classification" configs/pipeline.yaml

In [ ]:
!python -m indonesian_hoax_checker "Presiden Prabowo menandatangani Surat Keputusan baru di Jakarta." -v
!python -m indonesian_hoax_checker "Beredar pesan berantai yang mengeklaim Menteri Keuangan Purbaya akan menaikkan harga BBM subsidi pada 1 Desember 2026." -v
!python -m indonesian_hoax_checker "Pemerintah menghentikan bantuan" --task classify
!python -m indonesian_hoax_checker "Tautan Pendaftaran BLT UMKM 2026 Sebesar Rp50 Juta" --task classify
# Teks 200 kata atau lebih: classifier dipakai
!python -m indonesian_hoax_checker "Pemerintah daerah mengumumkan jadwal pembangunan jalan baru yang menghubungkan dua kecamatan di wilayah utara. Proyek tersebut direncanakan berlangsung selama delapan bulan dengan anggaran yang bersumber dari anggaran daerah. Warga diminta mengikuti pengaturan lalu lintas selama pengerjaan, karena sebagian ruas akan ditutup bergantian pada jam sibuk. Dinas pekerjaan umum menyiapkan jalur alternatif dan petugas yang mengatur arus kendaraan di persimpangan utama. Pemerintah juga menyampaikan bahwa pembayaran kepada kontraktor dilakukan bertahap sesuai kemajuan pekerjaan dan diawasi oleh inspektorat. Warga yang ingin menyampaikan keluhan dapat menghubungi kantor kecamatan setiap hari kerja atau mengirim pesan melalui kanal pengaduan resmi pemerintah daerah. Setelah proyek selesai, dinas akan menggelar peninjauan bersama warga untuk memastikan kualitas jalan sesuai ketentuan. Kepala dinas menjelaskan bahwa pembangunan dibagi menjadi tiga tahap, dimulai dari pelebaran badan jalan, dilanjutkan dengan pemasangan drainase di kedua sisi, lalu pengaspalan dan pemasangan rambu. Selama tahap pertama, pekerja akan bekerja pada pagi hari agar tidak mengganggu aktivitas sekolah dan pasar. Pihak kecamatan telah mengumpulkan perwakilan warga untuk membahas dampak proyek terhadap pedagang kecil di sepanjang jalan, dan disepakati bahwa lapak sementara akan disediakan di lahan kosong milik desa. Laporan kemajuan pekerjaan akan diumumkan setiap bulan di papan pengumuman kantor desa dan di laman resmi pemerintah daerah." -v
!python -m indonesian_hoax_checker "Beredar pesan berantai yang mengeklaim bahwa mulai bulan depan seluruh warga wajib menukarkan uang tunai mereka di bank tertentu, jika tidak maka uang tersebut akan dinyatakan tidak berlaku. Pesan itu meminta penerima segera menyebarkannya kepada keluarga dan teman agar tidak rugi. Disebutkan pula bahwa antrean akan sangat panjang sehingga setiap orang harus datang sejak pagi dan membawa semua uang simpanan di rumah. Pengirim pesan mengaku mendapat informasi dari orang dalam yang tidak mau disebutkan namanya, dan menambahkan bahwa pemerintah sengaja merahasiakan aturan ini agar masyarakat tidak panik. Siapa pun yang tidak mengikuti petunjuk dalam pesan ini disebut akan kehilangan seluruh uangnya. Dalam pesan yang sama ditulis bahwa tenggat penukaran hanya tiga hari dan tidak akan diperpanjang dengan alasan apa pun. Penerima juga diminta tidak menghubungi bank resmi karena nomor telepon yang tercantum di situs mereka disebut sudah disadap. Sebagai gantinya, pesan itu mencantumkan sebuah nomor yang katanya milik petugas khusus, lengkap dengan tautan formulir untuk mengisi data rekening dan kata sandi. Pesan ini kemudian diteruskan ke banyak grup percakapan tanpa ada yang memeriksa kebenarannya, dan sebagian orang bahkan mulai menarik tabungan mereka karena takut kehilangan uang. Beberapa pengguna media sosial menulis bahwa tetangga mereka sudah mengantre sejak subuh, padahal tidak ada pengumuman resmi apa pun dari pihak bank maupun otoritas keuangan mengenai penukaran uang tersebut." -v

## 9. Evaluasi panjang kalimat (opsional)
Nilai AUROC di dalam rentang jumlah kata yang sama. Butuh `indobert_results/` di Drive pribadi (`MyDrive/SEMESTER 7/NLP/Tubes/indobert_results`); lewati bila folder itu tidak ada.

In [ ]:
from google.colab import drive as gdrive
gdrive.mount("/content/drive")
SCORES = "/content/drive/MyDrive/SEMESTER 7/NLP/Tubes/indobert_results"
if os.path.isdir(SCORES):
    !pip -q install datasets
    !python classification/length_controlled_eval.py --scores-dir "{SCORES}" --mafindo
else:
    print("folder skor tidak ada di Drive ini; sel dilewati")